# File to Database Loader

## Reading CSV files from a directory
## Validating and processing the data
## writing the data into a sql database (mysql, sqlserver, postgresql, sqlite)
## handling the large files via chunking

In [33]:
from sqlalchemy import create_engine
username = 'root'
password = 'samurai28'
host = 'localhost'
port = 3306
database = 'avi'

engine = create_engine(f"mysql+pymysql://{username}:{password}@{host}:{port}/{database}")

In [34]:
import pandas as pd
df = pd.read_sql("select * from employees;", con=engine)

In [35]:
df

,emp_id,emp_name,department,salary,hire_date
0,1,Alice,Engineering,95000.0,2020-03-01
1,2,Bob,Engineering,88000.0,2019-07-15
2,3,Charlie,Engineering,102000.0,2021-01-10
3,4,Diana,Marketing,72000.0,2020-06-01
4,5,Eve,Marketing,68000.0,2021-03-15
5,6,Frank,Marketing,75000.0,2019-11-01
6,7,Grace,Finance,85000.0,2020-09-01
7,8,Hank,Finance,91000.0,2018-04-01
8,9,Ivy,Finance,78000.0,2022-01-15
9,10,Jack,Engineering,97000.0,2020-12-01


In [36]:
from sqlalchemy import create_engine
username = 'root'
password = 'samurai28'
host = 'localhost'
port = 3306
database = 'hr_db'

engine = create_engine(f"mysql+pymysql://{username}:{password}@{host}:{port}/{database}")

In [37]:
#Check if the filepath exists
import os

filedir = f'{os.getcwd()}/datafiles'
if not os.path.exists(filedir):
    os.makedirs(filedir)

deptcsv = f'{filedir}/departments.csv'
empcsv = f'{filedir}/employees.csv'
os.path.exists(deptcsv)

True

In [38]:
# Read data from csv file
import pandas as pd
df_dept = pd.read_csv(deptcsv)
df_emp = pd.read_csv(empcsv)
df.columns.tolist()

for col, dtype in df_emp.dtypes.items():
    print(col, dtype)

employee_id int64
first_name object
last_name object
email object
job_title object
hire_date object
salary int64
department_id int64
manager_id float64


In [ ]:
# Insert data into the MySQL table
df_dept.to_sql("departments", con=engine, if_exists='append', index=False)

# if_exists='replace' will try to drop the table

1. index=True will add an unwanted column
By default this writes your DataFrame's index as a new column (usually named index), which won't match the departments table schema you already defined in ddl.sql. Unless your DataFrame's index actually holds department_id

2. if_exists='replace' will drop your carefully-built schema
This tells pandas to DROP TABLE departments and recreate it from scratch, inferring column types from the DataFrame — which means:

    -> Your PRIMARY KEY, FOREIGN KEY, and NOT NULL constraints from ddl.sql are gone
    -> Column types become pandas' best guess (e.g. TEXT instead of VARCHAR(100), DOUBLE instead of DECIMAL(12,2))
    -> If any other table (like employees) already has a live FK pointing to departments, you'll hit the exact same OperationalError you just saw — MySQL will refuse to drop a table another table references

In [ ]:
# Check the loaded data
pd.read_sql("select * from departments;", con=engine)

,department_id,department_name,location,budget
0,1,Engineering,Bengaluru,5000000.0
1,2,Sales,Mumbai,3000000.0
2,3,Marketing,Delhi,2000000.0
3,4,Human Resources,Bengaluru,1200000.0
4,5,Finance,Pune,2500000.0
5,6,Customer Support,Hyderabad,1500000.0
6,7,IT Operations,Bengaluru,1800000.0
7,8,Legal,Delhi,900000.0


## Load data chunk wise

In [ ]:
df_emp.shape #(20, 9)
chunk_size = 5

df_emp.to_sql("employees", con=engine, if_exists='append', index=False, chunksize=chunk_size)

ValueError: Table 'employees' already exists.

In [ ]:
pd.read_sql("select * from employees;", con=engine)

,employee_id,first_name,last_name,email,job_title,hire_date,salary,department_id,manager_id
0,1,Rahul,Sharma,rahul.sharma@company.com,Account Manager,2019-04-02,51556.0,5,NaN
1,2,Priya,Verma,priya.verma@company.com,Sales Executive,2020-07-03,81579.0,2,NaN
2,3,Amit,Iyer,amit.iyer@company.com,Account Manager,2024-02-12,67790.0,7,NaN
3,4,Sneha,Reddy,sneha.reddy@company.com,Software Engineer,2018-05-03,69561.0,4,NaN
4,5,Vikram,Gupta,vikram.gupta@company.com,Sales Executive,2023-09-01,51956.0,4,NaN
5,6,Anita,Nair,anita.nair@company.com,QA Engineer,2024-02-11,154974.0,4,NaN
6,7,Karan,Patel,karan.patel@company.com,Support Engineer,2024-08-10,117926.0,1,6.0
7,8,Divya,Singh,divya.singh@company.com,Business Analyst,2022-09-28,134194.0,5,3.0
8,9,Arjun,Rao,arjun.rao@company.com,Business Analyst,2020-05-31,133236.0,2,1.0
9,10,Neha,Mehta,neha.mehta@company.com,Senior Software Engineer,2022-04-06,70353.0,6,2.0


## Load from multiple csv files

In [46]:
# Get all csv files from the folder

import os

filepath = f'{os.getcwd()}/datafiles'
files = [f for f in os.listdir(filepath) if f.endswith('.csv')]
files

['managers.csv',
 'olist_sellers_dataset.csv',
 'product_category_name_translation.csv',
 'olist_orders_dataset.csv',
 'olist_order_items_dataset.csv',
 'departments.csv',
 'employee_projects.csv',
 'olist_customers_dataset.csv',
 'olist_geolocation_dataset.csv',
 'olist_order_payments_dataset.csv',
 'employees.csv',
 'olist_order_reviews_dataset.csv',
 'projects.csv',
 'olist_products_dataset.csv']

In [64]:
# load files in their respective tables

from sqlalchemy import create_engine, inspect, text

usr = 'root'
password = os.environ['DB_PASSWORD']
host = 'localhost'
port = 3306
db = 'hr_db'

engine = create_engine(f'mysql+pymysql://{usr}:{password}@{host}:{port}/{db}')
inspector = inspect(engine)

for file in files:
    f = file.split('.')[0]
    if inspector.has_table(f):
        with engine.begin() as conn:
            conn.execute(text("SET FOREIGN_KEY_CHECKS = 0"))
            conn.execute(text(f'TRUNCATE {f}'))
            df = pd.read_csv(f'{filepath}/{file}')
            df.to_sql(f, con=conn, if_exists='append', index=False)
            conn.execute(text("SET FOREIGN_KEY_CHECKS = 1"))
        print(pd.read_sql(f"select '{f}' as tblname, count(*) from {f}", con=engine))
            #   print(f"Successful table load: {f}")
        
    

    tblname  count(*)
0  managers         6
       tblname  count(*)
0  departments         8
             tblname  count(*)
0  employee_projects        20
     tblname  count(*)
0  employees        20
    tblname  count(*)
0  projects        15
